# Long-Term Memory

This notebook shows a practical long-term memory setup with LangGraph:

- `SqliteSaver` for checkpointed thread state
- a persistent Chroma-based semantic memory index
- resume from checkpoint
- memory summarisation

LangGraph’s persistence docs say checkpointers provide short-term, thread-scoped memory while stores provide long-term, cross-thread memory. The checkpointer docs also show `SqliteSaver` as one of the provided checkpoint libraries, and the memory docs describe adding long-term memory, semantic search, and summarizing messages. The time-travel docs show how checkpoint history can be inspected and how a prior checkpoint can be resumed. 

## Learning goals

By the end of this notebook, you should be able to:

1. Persist thread state with `SqliteSaver`.
2. Store long-term memories in a semantic index.
3. Recall memories across turns and across threads.
4. Summarize conversation history into a compact memory.
5. Inspect checkpoint history and resume from a prior checkpoint.

## 1) Install packages

In [2]:
%pip install -qU langgraph langgraph-checkpoint-sqlite langchain langchain-core langchain-chroma chromadb sentence-transformers python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Environment variables

A Groq key is optional in this notebook. The memory mechanics work locally even without a model key.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))
print("LANGSMITH_API_KEY set:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))

GROQ_API_KEY set: True
LANGSMITH_API_KEY set: True
LANGSMITH_TRACING: true
LANGSMITH_PROJECT: lcel-groq-demo


## 3) Local persistence setup

In [3]:
import sqlite3
import shutil
from pathlib import Path
from datetime import datetime
from typing_extensions import TypedDict

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

CHECKPOINT_DB = DATA_DIR / "memory_checkpoints.db"
AUDIT_DB = DATA_DIR / "memory_audit.db"
MEMORY_DIR = Path("./chroma_memory_store")

if CHECKPOINT_DB.exists():
    CHECKPOINT_DB.unlink()
if AUDIT_DB.exists():
    AUDIT_DB.unlink()
if MEMORY_DIR.exists():
    shutil.rmtree(MEMORY_DIR)

print("Checkpoint DB:", CHECKPOINT_DB.resolve())
print("Audit DB:", AUDIT_DB.resolve())
print("Memory dir:", MEMORY_DIR.resolve())

Checkpoint DB: D:\personal_docs\course-ai\module-1\agenticai-with-langgraph\data\memory_checkpoints.db
Audit DB: D:\personal_docs\course-ai\module-1\agenticai-with-langgraph\data\memory_audit.db
Memory dir: D:\personal_docs\course-ai\module-1\agenticai-with-langgraph\chroma_memory_store


## 4) Semantic memory store

We use a persistent Chroma collection as a local semantic memory index.

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

memory_store = Chroma(
    collection_name="long_term_memory",
    persist_directory=str(MEMORY_DIR),
    embedding_function=embeddings,
)

print("Memory store ready")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4960.08it/s]


Memory store ready


## 5) Tiny audit table

This makes it easy to show what the graph saves over time.

In [5]:
conn = sqlite3.connect(AUDIT_DB)
cur = conn.cursor()

cur.execute(
    "CREATE TABLE memory_audit ("
    "id INTEGER PRIMARY KEY AUTOINCREMENT, "
    "user_id TEXT NOT NULL, "
    "thread_id TEXT NOT NULL, "
    "memory_text TEXT NOT NULL, "
    "created_at TEXT NOT NULL"
    ")"
)

conn.commit()
conn.close()

print("Audit DB created")

Audit DB created


## 6) Graph state

In [13]:
from typing import Annotated
from langgraph.graph.message import add_messages

class MemoryState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str
    summary: str
    memory_context: str
    response: str

## 7) Helper functions

The persistence docs distinguish short-term state in the checkpoint from durable memories in the store. The store docs also show semantic search over embedded memories. 

In [7]:
def latest_user_message(state: MemoryState) -> str:
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            return msg.content
        if isinstance(msg, dict) and msg.get("role") == "user":
            return msg.get("content", "")
    return ""

def lookup_memories(user_id: str, query: str, k: int = 3) -> str:
    results = memory_store.similarity_search(
        query=query,
        k=k,
        filter={"user_id": user_id},
    )
    if not results:
        return ""
    return "\n".join(f"- {doc.page_content}" for doc in results)

def summarize_history(messages: list, previous_summary: str = "") -> str:
    recent = []
    for msg in messages[-6:]:
        if isinstance(msg, HumanMessage):
            recent.append(f"user: {msg.content}")
        elif isinstance(msg, AIMessage):
            recent.append(f"assistant: {msg.content}")
        elif isinstance(msg, dict):
            role = msg.get("role", "unknown")
            recent.append(f"{role}: {msg.get('content', '')}")

    recent_text = "\n".join(recent)

    if previous_summary:
        return (previous_summary + " | " + recent_text)[:500]
    return recent_text[:500]

def store_memory(user_id: str, thread_id: str, memory_text: str, kind: str = "summary") -> None:
    doc = Document(
        page_content=memory_text,
        metadata={
            "user_id": user_id,
            "thread_id": thread_id,
            "kind": kind,
            "created_at": datetime.utcnow().isoformat(),
        },
    )
    memory_store.add_documents([doc])

    conn = sqlite3.connect(AUDIT_DB)
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO memory_audit (user_id, thread_id, memory_text, created_at) VALUES (?, ?, ?, ?)",
        (user_id, thread_id, memory_text, datetime.utcnow().isoformat()),
    )
    conn.commit()
    conn.close()

## 8) Graph nodes

We keep the graph simple:

1. retrieve relevant memories
2. answer the user
3. summarize the conversation
4. store the summary as long-term memory

The memory docs describe summarizing messages after enough conversation has accumulated, using the previous summary as context for the next one. 

In [14]:
from langchain_core.runnables import RunnableConfig

def retrieve_memory(state: MemoryState):
    query = latest_user_message(state)
    context = lookup_memories(state["user_id"], query, k=3)
    return {"memory_context": context}

def answer_user(state: MemoryState):
    question = latest_user_message(state)
    summary = state.get("summary", "")
    memory_context = state.get("memory_context", "")

    response = (
        f"Summary so far: {summary[:140]}\n"
        f"Relevant memories: {memory_context[:160]}\n"
        f"Answer to: {question}"
    )
    return {
        "response": response,
        "messages": [AIMessage(content=response)],
    }

def summarize_conversation(state: MemoryState):
    new_summary = summarize_history(state["messages"], state.get("summary", ""))
    return {"summary": new_summary}

def save_long_term_memory(state: MemoryState, config: RunnableConfig):
    user_id = state["user_id"]
    thread_id = config["configurable"].get("thread_id", "unknown-thread")
    memory_text = state.get("summary", "") or latest_user_message(state)
    store_memory(user_id, thread_id, memory_text, kind="summary")
    return {}

## 9) Build the graph with `SqliteSaver`

The checkpointer docs say a thread ID is required and that `graph.get_state_history(config)` gives the checkpoint history for that thread. They also say a prior `checkpoint_id` can be used for replay. 

In [11]:
builder = StateGraph(MemoryState)

builder.add_node("retrieve_memory", retrieve_memory)
builder.add_node("answer_user", answer_user)
builder.add_node("summarize_conversation", summarize_conversation)
builder.add_node("save_long_term_memory", save_long_term_memory)

builder.add_edge(START, "retrieve_memory")
builder.add_edge("retrieve_memory", "answer_user")
builder.add_edge("answer_user", "summarize_conversation")
builder.add_edge("summarize_conversation", "save_long_term_memory")
builder.add_edge("save_long_term_memory", END)

checkpointer = SqliteSaver(sqlite3.connect(str(CHECKPOINT_DB), check_same_thread=False))
graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled with SqliteSaver.")

Graph compiled with SqliteSaver.


## 10) First conversation turn

In [15]:
config = {"configurable": {"thread_id": "memory-thread-1"}}

turn1 = graph.invoke(
    {
        "messages": [HumanMessage(content="My name is Asha and I like concise answers.")],
        "user_id": "user-1",
        "summary": "",
        "memory_context": "",
        "response": "",
    },
    config=config,
)

turn1

C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_16700\2160606743.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),
C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_16700\2160606743.py:52: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (user_id, thread_id, memory_text, datetime.utcnow().isoformat()),


{'messages': [AIMessage(content='Summary so far: \nRelevant memories: - assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like concise answers.\nAnswer to: My name is Asha and I like concise answers.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'user_id': 'user-1',
 'summary': 'assistant: Summary so far: \nRelevant memories: - assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like concise answers.\nAnswer to: My name is Asha and I like concise answers.',
 'memory_context': '- assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like concise answers.',
 'response': 'Summary so far: \nRelevant memories: - assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like concise answers.\nAnswer to: My name is Asha and I like concise answers.'}

## 11) Second turn

In [16]:
turn2 = graph.invoke(
    {
        "messages": [HumanMessage(content="What should you remember about me?")],
        "user_id": "user-1",
        "summary": turn1.get("summary", ""),
        "memory_context": turn1.get("memory_context", ""),
        "response": "",
    },
    config=config,
)

turn2

C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_16700\2160606743.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),
C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_16700\2160606743.py:52: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (user_id, thread_id, memory_text, datetime.utcnow().isoformat()),


{'messages': [AIMessage(content='Summary so far: assistant: Summary so far: \nRelevant memories: - assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like conci\nRelevant memories: - assistant: Summary so far: \nRelevant memories: - assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like concise answers.\nAnswer\nAnswer to: What should you remember about me?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'user_id': 'user-1',
 'summary': 'assistant: Summary so far: \nRelevant memories: - assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like concise answers.\nAnswer to: My name is Asha and I like concise answers. | assistant: Summary so far: assistant: Summary so far: \nRelevant memories: - assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like conci\nRelevant memories: - assistant: Summary so far: \nRelevant memories: 

## 12) View checkpoint history

The newest checkpoint appears first in the list.

In [17]:
history = list(graph.get_state_history(config))
print("Checkpoint count:", len(history))

for i, snap in enumerate(history[:3], 1):
    print(f"--- Snapshot {i} ---")
    print("checkpoint_id:", snap.config["configurable"]["checkpoint_id"])
    print("next:", snap.next)
    print("summary:", snap.values.get("summary", "")[:120])
    print()

Checkpoint count: 12
--- Snapshot 1 ---
checkpoint_id: 1f16a1d8-da8c-69ed-800a-cb7a6bc7f2ae
next: ()
summary: assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answer to: My name is A

--- Snapshot 2 ---
checkpoint_id: 1f16a1d8-d9f2-64c8-8009-5cc483ff9b85
next: ('save_long_term_memory',)
summary: assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answer to: My name is A

--- Snapshot 3 ---
checkpoint_id: 1f16a1d8-d9eb-6679-8008-051104917656
next: ('summarize_conversation',)
summary: assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answer to: My name is A



## 13) Resume from a prior checkpoint

The time-travel docs describe resuming from a selected checkpoint and exploring an alternate path. The checkpointer docs also explain that replay re-executes nodes after the chosen checkpoint. 

In [18]:
if len(history) >= 2:
    old_checkpoint_id = history[-2].config["configurable"]["checkpoint_id"]

    replay_config = {
        "configurable": {
            "thread_id": "memory-thread-1",
            "checkpoint_id": old_checkpoint_id,
        }
    }

    replayed = graph.invoke(
        {
            "messages": [HumanMessage(content="If I ask again, what do you remember about me?")],
            "user_id": "user-1",
            "summary": history[-2].values.get("summary", ""),
            "memory_context": history[-2].values.get("memory_context", ""),
            "response": "",
        },
        config=replay_config,
    )

    replayed
else:
    print("Not enough checkpoints yet to replay.")

C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_16700\2160606743.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),
C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_16700\2160606743.py:52: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (user_id, thread_id, memory_text, datetime.utcnow().isoformat()),


## 14) Inspect the semantic memory store

In [19]:
direct_hits = memory_store.similarity_search(
    "What does the user like?",
    k=3,
    filter={"user_id": "user-1"},
)

for i, doc in enumerate(direct_hits, 1):
    print(f"--- Memory hit {i} ---")
    print(doc.page_content)
    print(doc.metadata)
    print()

--- Memory hit 1 ---
assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answer to: My name is Asha and I like concise answers.
- assistant: Summary so far: 
Relevant memories: - assi
Answer to: If I ask again, what do you remember about me?
{'thread_id': 'unknown-thread', 'user_id': 'user-1', 'kind': 'summary', 'created_at': '2026-06-17T07:24:33.891430'}

--- Memory hit 2 ---
assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answer to: My name is Asha and I like concise answers.
Answer to: My name is Asha and I like concise answers. | assistant: Summary so far: assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answer to: My name is Asha and I like conci
Relevant memories: - assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answe
{'created_at': '2026-06-17T07:24:18.782863', 'kind': 'summary', 'thread_id

## 15) Memory summarisation

In [20]:
print("Current summary:")
print(turn2.get("summary", ""))

print("\nAudit rows:")
conn = sqlite3.connect(AUDIT_DB)
cur = conn.cursor()
cur.execute("SELECT user_id, thread_id, memory_text FROM memory_audit")
for row in cur.fetchall():
    print(row)
conn.close()

Current summary:
assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answer to: My name is Asha and I like concise answers.
Answer to: My name is Asha and I like concise answers. | assistant: Summary so far: assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answer to: My name is Asha and I like conci
Relevant memories: - assistant: Summary so far: 
Relevant memories: - assistant: Summary so far: 
Relevant memories: 
Answe

Audit rows:
('user-1', 'unknown-thread', 'assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like concise answers.')
('user-1', 'unknown-thread', 'assistant: Summary so far: \nRelevant memories: - assistant: Summary so far: \nRelevant memories: \nAnswer to: My name is Asha and I like concise answers.\nAnswer to: My name is Asha and I like concise answers.')
('user-1', 'unknown-thread', 'assistant: Summary so far: \nRelevant memories: - assistant

## Key takeaways

- Checkpointers keep thread-scoped short-term state.
- Stores keep durable long-term memory across threads.
- Semantic search lets you recall memories by meaning.
- Summarization keeps old conversation context compact.
- `graph.get_state_history(config)` and a prior `checkpoint_id` let you inspect and resume a conversation. 

## References

- Persistence: https://docs.langchain.com/oss/python/langgraph/persistence
- Checkpointers: https://docs.langchain.com/oss/python/langgraph/checkpointers
- Add memory: https://docs.langchain.com/oss/python/langgraph/add-memory
- Stores: https://docs.langchain.com/oss/python/langgraph/stores
- Time travel: https://docs.langchain.com/oss/python/langchain/frontend/time-travel